In [ ]:
root_path = "/gpfs/home/pl2948/P_KNN/revision"

# Import data

In [ ]:
select_column = [
    'BayesDel_nsfp33a_noAF',
    'MutPred2.0_score',
    'REVEL_score',
    'VEST4_score', 
    'AlphaMissense_score',
    'ESM1b_score',
    'VARITY_R_score',
    'ClinVar_annotation'
]

## ClinVar 2019

In [ ]:
import pandas as pd
import os

ClinVar_2019 = pd.read_csv(os.path.join(root_path, "data", "ClinVar2019_best7.txt"), sep='\t')
print(ClinVar_2019.columns)
display(ClinVar_2019)

In [ ]:
calibration_data = ClinVar_2019.rename(columns={'clnsig': 'ClinVar_annotation'})
# del ClinVar_2019

ClinVar_ann = {
    'Benign/Likely_benign': 0, 
    'Likely_benign': 0, 
    'Pathogenic': 1,
    'Likely_pathogenic': 1, 
    'Benign': 0, 
    'Pathogenic/Likely_pathogenic': 1
    }

column_map = {'BayesDel_nsfp33a_noAF': 'BayesDel_noAF_score', 
              'MutPred2.0_score': "MutPred2_score", 
              'ESM1b_score': 'ESM1b_score',
              'AlphaMissense_score': 'AlphaMissense_score', 
              'VARITY_R_score': 'VARITY_R_score',
              'VEST4_score': 'VEST4_score', 
              'REVEL_score': 'REVEL_score',
}

calibration_data['ClinVar_annotation'] = calibration_data['ClinVar_annotation'].map(ClinVar_ann)
calibration_data = calibration_data[calibration_data['ClinVar_annotation'].notna()]
# calibration_data = calibration_data[select_column]

print(len(calibration_data))

for col in select_column[:-1]:
    calibration_data[col] = pd.to_numeric(calibration_data[col], errors='coerce')
    # if col in ['SIFT_score', 'SIFT4G_score', 'PROVEAN_score', 'MutationTaster_trees_benign', 'ESM1b_score', 'bStatistic']:
    #     calibration_data[col] *=-1

calibration_data = calibration_data.rename(columns=column_map)
calibration_data = calibration_data.reset_index(drop=True)

display(calibration_data)
print(calibration_data.isna().sum()/len(calibration_data))

calibration_data.to_csv(os.path.join(root_path, "data", "calibration_data_ClinGen7.csv"))

In [ ]:
calibration_data = ClinVar_2019.rename(columns={'clnsig': 'ClinVar_annotation'})
del ClinVar_2019

ClinVar_ann = {
    'Benign/Likely_benign': 0, 
    'Likely_benign': 0, 
    'Pathogenic': 1,
    'Likely_pathogenic': 1, 
    'Benign': 0, 
    'Pathogenic/Likely_pathogenic': 1
    }

calibration_data['ClinVar_annotation'] = calibration_data['ClinVar_annotation'].map(ClinVar_ann)
calibration_data = calibration_data[calibration_data['ClinVar_annotation'].notna()]
calibration_data = calibration_data[select_column]

print(len(calibration_data))

for col in select_column[:-1]:
    calibration_data[col] = pd.to_numeric(calibration_data[col], errors='coerce')
    if col in ['SIFT_score', 'SIFT4G_score', 'PROVEAN_score', 'MutationTaster_trees_benign', 'ESM1b_score', 'bStatistic']:
        calibration_data[col] *=-1

calibration_data = calibration_data.reset_index(drop=True)

display(calibration_data)
print(calibration_data.isna().sum()/len(calibration_data))

In [ ]:
print("Pathogenic", sum(calibration_data['ClinVar_annotation']==1))
print("Benign", sum(calibration_data['ClinVar_annotation']==0))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, auc
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

def calculate_roc_and_auc(y_true, y_score, label):
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    valid_indices = ~np.isnan(y_score) & ~np.isnan(y_true)
    y_true_cleaned = y_true[valid_indices]
    y_score_cleaned = y_score[valid_indices]

    fpr, tpr, _ = roc_curve(y_true_cleaned, y_score_cleaned)
    roc_auc = auc(fpr, tpr)
    return fpr, tpr, roc_auc, label

roc_curves = []

for col in select_column[:-1]:  # Exclude 'ClinVar_annotation'
    roc_curves.append(calculate_roc_and_auc(
        calibration_data['ClinVar_annotation'], calibration_data[col], col))

# Plot ROC curves
plt.figure(figsize=(8, 6))
for fpr, tpr, roc_auc, label in roc_curves:
    plt.plot(fpr, tpr, label=f'{label} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], color='gray', linestyle='--')  # Reference diagonal line
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc='lower right')
# plt.grid()
plt.show()

## ClinVar 2020

In [ ]:
import pandas as pd
import os

ClinVar_2020 = pd.read_csv(os.path.join(root_path, "data", "ClinVar2020_best7.txt"), sep='\t')
print(ClinVar_2020.columns)
display(ClinVar_2020)

In [ ]:
test_data = ClinVar_2020.rename(columns={'clnsig': 'ClinVar_annotation'})
del ClinVar_2020

ClinVar_ann = {
    'Benign/Likely_benign': 0, 
    'Likely_benign': 0, 
    'Pathogenic': 1,
    'Likely_pathogenic': 1, 
    'Benign': 0, 
    'Pathogenic/Likely_pathogenic': 1
    }

test_data['ClinVar_annotation'] = test_data['ClinVar_annotation'].map(ClinVar_ann)
test_data = test_data[test_data['ClinVar_annotation'].notna()]
test_data = test_data[select_column]

print(len(test_data))

for col in select_column[:-1]:
    test_data[col] = pd.to_numeric(test_data[col], errors='coerce')
    if col in ['SIFT_score', 'SIFT4G_score', 'PROVEAN_score', 'MutationTaster_trees_benign', 'ESM1b_score', 'bStatistic']:
        test_data[col] *=-1

test_data = test_data.reset_index(drop=True)

display(test_data)
print(test_data.isna().sum()/len(test_data))

In [ ]:
print("Pathogenic", sum(test_data['ClinVar_annotation']==1))
print("Benign", sum(test_data['ClinVar_annotation']==0))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, auc
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

roc_curves = []

for col in select_column[:-1]:  # Exclude 'ClinVar_annotation'
    roc_curves.append(calculate_roc_and_auc(
        test_data['ClinVar_annotation'], test_data[col], col))

plt.figure(figsize=(8, 6))
for fpr, tpr, roc_auc, label in roc_curves:
    plt.plot(fpr, tpr, label=f'{label} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], color='gray', linestyle='--')  # Reference diagonal line
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc='lower right')
# plt.grid()
plt.show()

## gnomAD set

In [ ]:
import pandas as pd
import os

gnomAD_set = pd.read_csv(os.path.join(root_path, 'data', "GnomADSet_best7.txt"), sep='\t')
print(gnomAD_set.columns)
display(gnomAD_set)

In [ ]:
gnomAD_set = gnomAD_set[select_column[:-1]].copy()

print(len(gnomAD_set))

for col in select_column[:-1]:
    gnomAD_set[col] = pd.to_numeric(gnomAD_set[col], errors='coerce')
    if col in ['SIFT_score', 'SIFT4G_score', 'PROVEAN_score', 'MutationTaster_trees_benign', 'ESM1b_score', 'bStatistic']:
        gnomAD_set[col] *=-1

gnomAD_set = gnomAD_set.reset_index(drop=True)
    
display(gnomAD_set)
print(gnomAD_set.isna().sum()/len(gnomAD_set))

# Single tool calibration

In [ ]:
import copy
import numpy as np

calibration_feature = calibration_data[select_column[:-1]].to_numpy()
test_feature = test_data[select_column[:-1]].to_numpy()
regularization_feature = gnomAD_set[select_column[:-1]].to_numpy()

calibration_label = calibration_data[select_column[-1]].to_numpy()
calibration_label_bk = copy.deepcopy(calibration_label)
test_label = test_data[select_column[-1]].to_numpy()
test_label_bk = copy.deepcopy(test_label)

print(calibration_feature.shape, test_feature.shape, regularization_feature.shape)

In [ ]:
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0) 
    total_memory_bytes = props.total_memory
    total_memory_gb = total_memory_bytes / (1024**3)
    print(f"GPU 名稱: {props.name}")
    print(f"總 VRAM 大小: {total_memory_gb:.2f} GB")
else:
    print("未檢測到可用的 CUDA GPU。")

In [ ]:
import torch
import gc
from P_KNN_GPU import get_bootstrap_KNN_score_gpu, get_P_KNN_ACMG_score_1D, evaluate_result_1D
import copy
import os

#Parameter setting
Pprior = 0.0441; # Prior probability of pathogenicity (changes w/ c)
w_calibration=None
n_calibration_in_window=100;   # minimum number of clinvar variants in a local window
frac_regularization_in_window=0.03
batch_size = 512 # for 16GB VRAM, use 512
normalization= None
impute = False
mi_scaling = False
n_bootstrap = 100

p_value = 0.05
logbase = 1124

combine_data = pd.DataFrame([])

best_mean_evidence_strength = 0

for i in range(len(select_column)-1):   
    condition_string = select_column[i]
    select_feature = i

    valid_calibration_idx = ~np.isnan(calibration_feature[:,select_feature]) 
    calibration_array = calibration_feature[valid_calibration_idx][:,select_feature].reshape(-1, 1)
    calibration_label = copy.deepcopy(calibration_label_bk[valid_calibration_idx])

    valid_test_idx = ~np.isnan(test_feature[:,select_feature])
    test_array = test_feature[valid_test_idx][:,select_feature].reshape(-1, 1)
    test_label = copy.deepcopy(test_label_bk[valid_test_idx])

    valid_regularization_idx = ~np.isnan(regularization_feature[:,select_feature])
    regularization_array = regularization_feature[valid_regularization_idx][:,select_feature].reshape(-1, 1)

    print("")
    print(f"Tool {condition_string}")
    print(f"calibration size: {calibration_array.shape[0]}, test size: {test_array.shape[0]}")

    torch.cuda.empty_cache()
    gc.collect()

    # test_results_array = get_bootstrap_KNN_score_gpu(calibration_array, test_array, regularization_array, 
    #                                                  calibration_label, Pprior, w_calibration, 
    #                                                  n_calibration_in_window, frac_regularization_in_window, 
    #                                                  normalization, impute, mi_scaling, n_bootstrap, batch_size)

    # np.save(os.path.join(root_path, 'PKNNMatrix', f'PKNNMatrix_{condition_string}.npy'), test_results_array)

    test_results_array = np.load(os.path.join(root_path, 'PKNNMatrix', f'PKNNMatrix_{condition_string}.npy'))

    P_KNN_pathogenic, P_KNN_benign, ACMG_scores = get_P_KNN_ACMG_score_1D(test_results_array, test_array, p_value, Pprior, logbase)

    evidence_strength_data, pathogenic_calibration_dict, benign_calibration_dict = evaluate_result_1D(test_results_array,
                                                                                                      test_array,
                                                                                                      test_label, 
                                                                                                      p_value, 
                                                                                                      Pprior, 
                                                                                                      logbase, 
                                                                                                      category = condition_string, 
                                                                                                      show_plot=True, 
                                                                                                      save_name=condition_string
                                                                                                          )

    test_data.loc[valid_test_idx, f"ACMGLLR_{condition_string}"] = ACMG_scores
    test_data.loc[valid_test_idx, f"P-KNN_Pathogenic_{condition_string}"] = P_KNN_pathogenic

    combine_data = pd.concat([combine_data, evidence_strength_data], ignore_index=True)

    mean_evidence_strength = evidence_strength_data[evidence_strength_data['Label']=='Pathogenic variants']['Score'].mean() + evidence_strength_data[evidence_strength_data['Label']=='Benign variants']['Score'].mean()
    if mean_evidence_strength >  best_mean_evidence_strength:
        best_mean_evidence_strength = mean_evidence_strength
        best_tool = i
        
    print("pathogenic evidence mean", f"{evidence_strength_data[evidence_strength_data['Label']=='Pathogenic variants']['Score'].mean():.3f}")
    print("benign evidence mean", f"{evidence_strength_data[evidence_strength_data['Label']=='Benign variants']['Score'].mean():.3f}")

print(f"Best Tool {select_column[best_tool]}")

## Try beta for binomial distribution

In [ ]:
import numpy as np
from scipy.stats import beta as beta_dist

def weighted_score_with_binom_ci(p_array, n_array, w, p_value=0.05):
    """
    使用 Beta(1,1) Prior (Laplace smoothing) 計算校正後的點估計與信賴區間。
    若該 bin 沒有任何數據 (t=0)，則回傳 NaN。
    """
    p_array = np.asarray(p_array)
    n_array = np.asarray(n_array)
    shape = p_array.shape

    scores = np.full(shape, np.nan)
    ci_lower = np.full(shape, np.nan)
    ci_upper = np.full(shape, np.nan)

    t_array = p_array + n_array

    # 定義從 Bayesian 點估計到後驗概率的權重轉換函數
    def weighted(pi):
        return pi / (pi + w * (1 - pi))

    it = np.nditer(p_array, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        p = p_array[idx]
        t = t_array[idx]

        # 關鍵修改：只有在 t > 0 時才進行計算，否則保留 NaN
        if t > 0:
            alpha_prior = 1
            beta_prior = 1
            
            # 1. 點估計 (Laplace Smoothing)
            pi_hat_smoothed = (p + alpha_prior) / (t + alpha_prior + beta_prior)
            
            # 2. 信賴區間 (Bayesian Credible Interval)
            ci_low = beta_dist.ppf(p_value / 2, p + alpha_prior, (t - p) + beta_prior)
            ci_up = beta_dist.ppf(1 - p_value / 2, p + alpha_prior, (t - p) + beta_prior)

            scores[idx] = weighted(pi_hat_smoothed)
            ci_lower[idx] = weighted(ci_low)
            ci_upper[idx] = weighted(ci_up)
        
        it.iternext()
    
    return scores, ci_lower, ci_upper

In [ ]:
def evaluate_result_1D(test_results_array, test_score, test_label, p_value, Pprior, logbase=None, category = None, show_plot=True, save_name=None):
    if logbase is None:
        logbase = get_logbase(Pprior)
    
    w_test = (1 - Pprior) * sum(test_label == 1) / (sum(test_label == 0) * Pprior)  
    
    # Calculate the pathogenic and benign threshold for evidence level
    Post_p = np.zeros(4) 
    Post_b = np.zeros(4)

    for j in range(4):
        Post_p[j] = logbase ** (1 / 2 ** j) * Pprior / ((logbase ** (1 / 2 ** j) - 1) * Pprior + 1)
        Post_b[j] = (logbase ** (1 / 2 ** j)) * (1 - Pprior) / (((logbase ** (1 / 2 ** j)) - 1) * (1 - Pprior) + 1)

    tool_score = test_score.flatten()

    # Deal with pathogenic
    sorted_indices = np.argsort(-tool_score) # from largest to smallest
    sorted_pathogenic_prob = test_results_array[sorted_indices]

    sorted_pathogenic_prob = np.minimum.accumulate(sorted_pathogenic_prob, axis=0)
    
    index = int(np.ceil(p_value*test_results_array.shape[1]))
    CI_pathogenic_prob = np.sort(sorted_pathogenic_prob, axis=1)[:, index-1]

    P_KNN_pathogenic = np.zeros_like(CI_pathogenic_prob)
    P_KNN_pathogenic[sorted_indices] = CI_pathogenic_prob

    # Deal with benign
    sorted_indices = np.argsort(tool_score) # from smallest to largest
    sorted_benign_prob = test_results_array[sorted_indices]
    sorted_benign_prob = 1- sorted_benign_prob

    sorted_benign_prob = np.minimum.accumulate(sorted_benign_prob, axis=0)
    
    CI_benign_prob = np.sort(sorted_benign_prob, axis=1)[:, index-1]

    P_KNN_benign = np.zeros_like(CI_benign_prob)
    P_KNN_benign[sorted_indices] = CI_benign_prob

    # Calculate ACMG scores
    ACMG_pathogenic_scores = Probability2ACMG_score(P_KNN_pathogenic, Pprior, logbase)
    ACMG_benign_scores = Probability2ACMG_score(P_KNN_benign, 1-Pprior, logbase)

    ACMG_pathogenic_scores[ACMG_pathogenic_scores < 0] = 0
    ACMG_benign_scores[ACMG_benign_scores < 0] = 0
    ACMG_pathogenic_scores[ACMG_pathogenic_scores > 8] = 8
    ACMG_benign_scores[ACMG_benign_scores > 8] = 8

    ACMG_scores = ACMG_pathogenic_scores - ACMG_benign_scores

    # Divide the results into pathogenic and benign based on test_label
    Pathogenic_P_KNN_pathogenic = P_KNN_pathogenic[test_label == 1]
    Benign_P_KNN_pathogenic = P_KNN_pathogenic[test_label == 0]
    Pathogenic_P_KNN_benign = P_KNN_benign[test_label == 1]
    Benign_P_KNN_benign = P_KNN_benign[test_label == 0]
    Pathogenic_ACMG_scores = ACMG_scores[test_label == 1]
    Benign_ACMG_scores = ACMG_scores[test_label == 0]
    
    # Print correct and incorrect assignment of pathogenic and benign variants
    print("Pathogenic evidence")
    accumulate_pathogenic_count = 0
    accumulate_benign_count = 0
    for ACMGevidence, threshold in zip(["+8", "+4", "+2", "+1"], Post_p):
        pathogenic_count = (Pathogenic_P_KNN_pathogenic > threshold).sum() - accumulate_pathogenic_count
        benign_count = (Benign_P_KNN_pathogenic > threshold).sum() - accumulate_benign_count
        accumulate_pathogenic_count += pathogenic_count
        accumulate_benign_count += benign_count
        print(f"  Evidence score {ACMGevidence} Probability threshold {threshold:.3f}:")
        print(f"    Pathogenic {pathogenic_count} ({pathogenic_count/(len(Pathogenic_P_KNN_pathogenic)):.2%}) pathogenic variants")
        print(f"    Benign {benign_count} ({benign_count/(len(Benign_P_KNN_pathogenic)):.2%}) error benign variants")
        print(f"    Weighted correct rate: {pathogenic_count/(pathogenic_count+benign_count*w_test):.2%}")

    print("Benign evidence")
    accumulate_pathogenic_count = 0
    accumulate_benign_count = 0
    for ACMGevidence, threshold in zip(["+8", "+4", "+2", "+1"], Post_b):
        pathogenic_count = (Pathogenic_P_KNN_benign > threshold).sum() - accumulate_pathogenic_count
        benign_count = (Benign_P_KNN_benign > threshold).sum() - accumulate_benign_count
        accumulate_pathogenic_count += pathogenic_count
        accumulate_benign_count += benign_count
        print(f"  Evidence score {ACMGevidence} Probability threshold {threshold:.3f}:")
        print(f"    Benign {benign_count} ({benign_count/(len(Benign_P_KNN_benign)):.2%}) benign variants")
        print(f"    Pathogenic {pathogenic_count} ({pathogenic_count/(len(Pathogenic_P_KNN_benign)):.2%}) error pathogenic variants")
        print(f"    Weighted correct rate: {benign_count*w_test/(pathogenic_count+benign_count*w_test):.2%}")

    # Evidence violin plot
    evidence_strength_data = pd.DataFrame({
        "Score": np.concatenate([-Benign_ACMG_scores, Pathogenic_ACMG_scores]),
        "Label": ["Benign variants"] * len(Benign_ACMG_scores) + ["Pathogenic variants"] * len(Pathogenic_ACMG_scores),
        "Category": [category] * (len(Benign_ACMG_scores) + len(Pathogenic_ACMG_scores))
        })

    if show_plot:
        import matplotlib.pyplot as plt
        import seaborn as sns
        sns.set(style="ticks")
    
        ax = sns.violinplot(
            x="Category",
            y="Score", 
            hue="Label",   # Pathogenic and Benign 
            data=evidence_strength_data, 
            split=True,   # on each side of violin
            inner=None, 
            palette={"Pathogenic variants": "red", "Benign variants": "blue"},
            alpha=0.6, 
            density_norm='area'
        )

        dark_gray = '0.3'

        sns.boxplot(
            x="Category", y="Score", hue="Label", data=evidence_strength_data,
            showcaps=True,  
            showfliers=False,
            palette={"Pathogenic variants": "pink", "Benign variants": "#bae0f5"},
            width=0.1, dodge=True, ax=ax,
            whiskerprops={'color': dark_gray, 'linewidth': 1.5, 'zorder': 2},
            capprops={'color': dark_gray, 'linewidth': 1.5},
            medianprops={'color': dark_gray, 'linewidth': 1.5},
            boxprops={'zorder': 2, 'edgecolor': dark_gray, 'linewidth': 1.5},

        )
        
        plt.xlabel("")
        plt.ylabel("Evidence strength (LLR)", fontsize=14)
        plt.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), fontsize=14)
        sns.despine(top=True, right=True)
        
        if save_name:
            plt.savefig(f"{save_name}_ACMGevidence.svg", format="svg", bbox_inches='tight')
        plt.show()

    # Pathogenic Calibration Plot with Confidence Interval
    num_bins = 10
    bins = np.linspace(0, 1, num_bins + 1)
    
    pathogenic_counts, _ = np.histogram(Pathogenic_P_KNN_pathogenic, bins=bins)
    benign_counts, _ = np.histogram(Benign_P_KNN_pathogenic, bins=bins)

    pathogenic_ratios, ci_lower, ci_upper = weighted_score_with_binom_ci(pathogenic_counts, benign_counts, w_test, p_value)

    bin_centers = (bins[:-1] + bins[1:]) / 2
    valid_mask = ~np.isnan(pathogenic_ratios) & ~np.isnan(ci_lower) & ~np.isnan(ci_upper)

    pathogenic_calibration_dict = {}

    pathogenic_calibration_dict['bin_centers'] = bin_centers[valid_mask]
    pathogenic_calibration_dict['ratios'] = pathogenic_ratios[valid_mask]
    pathogenic_calibration_dict['ci_lower'] = ci_lower[valid_mask]
    pathogenic_calibration_dict['ci_upper'] = ci_upper[valid_mask]


    bins = np.linspace(0.95, 1, num_bins + 1)

    pathogenic_counts, _ = np.histogram(Pathogenic_P_KNN_benign, bins=bins)
    benign_counts, _ = np.histogram(Benign_P_KNN_benign, bins=bins)
    
    benign_ratios, ci_lower, ci_upper = weighted_score_with_binom_ci(benign_counts, pathogenic_counts, 1/w_test, p_value)

    bin_centers = (bins[:-1] + bins[1:]) / 2
    valid_mask = ~np.isnan(benign_ratios) & ~np.isnan(ci_lower) & ~np.isnan(ci_upper)

    benign_calibration_dict = {}
    benign_calibration_dict['bin_centers'] = bin_centers[valid_mask]
    benign_calibration_dict['ratios'] = benign_ratios[valid_mask]
    benign_calibration_dict['ci_lower'] = ci_lower[valid_mask]
    benign_calibration_dict['ci_upper'] = ci_upper[valid_mask]

    if show_plot:
        # Plot the calibration curve for pathogenic
        plt.figure(figsize=(6, 6))
        plt.plot(pathogenic_calibration_dict['bin_centers'], pathogenic_calibration_dict['ratios'], 
                 'o-', color='red', label='Frequency of pathogenic variants')
        plt.fill_between(pathogenic_calibration_dict['bin_centers'], 
                         pathogenic_calibration_dict['ci_lower'], pathogenic_calibration_dict['ci_upper'], 
                         color='red', alpha=0.2, label='95% Confidence Interval')
        plt.xlabel("Posterior probability (pathogenic)", fontsize=14)
        plt.ylabel("Frequency of pathogenic variants", fontsize=14)
        plt.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), fontsize=14)
        
        x_vals = np.linspace(0, 1, 100)
        plt.plot(x_vals, x_vals, label="y = x", color='red', linestyle='--', alpha=0.2)
        
        for threshold in Post_p:
            plt.axvline(x=threshold, color='orange', linestyle='--', label=f'X = {threshold}', alpha=0.2)
            plt.axhline(y=threshold, color='orange', linestyle='--', label=f'Y = {threshold}', alpha=0.2)

        sns.despine(top=True, right=True)   
        if save_name:
            plt.savefig(f"{save_name}_P_Calibration.svg", format="svg", bbox_inches='tight')

        plt.show()

        # Plot the calibration curve for benign
        plt.figure(figsize=(6, 6))
        plt.plot(benign_calibration_dict['bin_centers'], benign_calibration_dict['ratios'],
                 'o-', color='blue', label='Frequency of benign variants')
        plt.fill_between(benign_calibration_dict['bin_centers'],
                         benign_calibration_dict['ci_lower'], benign_calibration_dict['ci_upper'],
                        color='blue', alpha=0.2, label='95% Confidence Interval')
        plt.xlabel("Posterior probability (benign)", fontsize=14)
        plt.ylabel("Frequency of benign variants", fontsize=14)
        plt.ylim(0.95, 1)
        plt.xlim(0.95, 1)
        plt.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), fontsize=14)

        x_vals = np.linspace(0.95, 1, 100)
        plt.plot(x_vals, x_vals, label="y = x", color='blue', linestyle='--', alpha=0.2)
        
        for threshold in Post_b:
            plt.axvline(x=threshold, color='cyan', linestyle='--', label=f'X = {threshold}', alpha=0.2)
            plt.axhline(y=threshold, color='cyan', linestyle='--', label=f'Y = {threshold}', alpha=0.2)

        sns.despine(top=True, right=True)
        if save_name:
            plt.savefig(f"{save_name}_B_Calibration.svg", format="svg", bbox_inches='tight')

        plt.show()

    return evidence_strength_data, pathogenic_calibration_dict, benign_calibration_dict


In [ ]:
def Probability2ACMG_score(Ppost, Pprior, logbase=None):
    if logbase is None:
        logbase = get_logbase(Pprior)

    Likelihood_ratio = Ppost * (1 - Pprior) / ((1 - Ppost) * Pprior)
    ACMG_scores = 8 * np.log(Likelihood_ratio) / np.log(logbase)

    return ACMG_scores

In [ ]:
import torch
import gc
from P_KNN_GPU import get_bootstrap_KNN_score_gpu , get_P_KNN_ACMG_score_1D #, evaluate_result_1D
import copy
import os

#Parameter setting
Pprior = 0.0441; # Prior probability of pathogenicity (changes w/ c)
w_calibration=None
n_calibration_in_window=100;   # minimum number of clinvar variants in a local window
frac_regularization_in_window=0.03
batch_size = 512 # for 16GB VRAM, use 512
normalization= None
impute = False
mi_scaling = False
n_bootstrap = 100

p_value = 0.05
logbase = 1124

combine_data = pd.DataFrame([])

best_mean_evidence_strength = 0

for i in range(len(select_column)-1):   
    condition_string = select_column[i]
    select_feature = i

    valid_calibration_idx = ~np.isnan(calibration_feature[:,select_feature]) 
    calibration_array = calibration_feature[valid_calibration_idx][:,select_feature].reshape(-1, 1)
    calibration_label = copy.deepcopy(calibration_label_bk[valid_calibration_idx])

    valid_test_idx = ~np.isnan(test_feature[:,select_feature])
    test_array = test_feature[valid_test_idx][:,select_feature].reshape(-1, 1)
    test_label = copy.deepcopy(test_label_bk[valid_test_idx])

    valid_regularization_idx = ~np.isnan(regularization_feature[:,select_feature])
    regularization_array = regularization_feature[valid_regularization_idx][:,select_feature].reshape(-1, 1)

    print("")
    print(f"Tool {condition_string}")
    print(f"calibration size: {calibration_array.shape[0]}, test size: {test_array.shape[0]}")

    torch.cuda.empty_cache()
    gc.collect()

    # test_results_array = get_bootstrap_KNN_score_gpu(calibration_array, test_array, regularization_array, 
    #                                                  calibration_label, Pprior, w_calibration, 
    #                                                  n_calibration_in_window, frac_regularization_in_window, 
    #                                                  normalization, impute, mi_scaling, n_bootstrap, batch_size)

    # np.save(os.path.join(root_path, 'PKNNMatrix', f'PKNNMatrix_{condition_string}.npy'), test_results_array)

    test_results_array = np.load(os.path.join(root_path, 'PKNNMatrix', f'PKNNMatrix_{condition_string}.npy'))

    P_KNN_pathogenic, P_KNN_benign, ACMG_scores = get_P_KNN_ACMG_score_1D(test_results_array, test_array, p_value, Pprior, logbase)

    evidence_strength_data, pathogenic_calibration_dict, benign_calibration_dict = evaluate_result_1D(test_results_array,
                                                                                                      test_array,
                                                                                                      test_label, 
                                                                                                      p_value, 
                                                                                                      Pprior, 
                                                                                                      logbase, 
                                                                                                      category = condition_string, 
                                                                                                      show_plot=True, 
                                                                                                      save_name=f"{condition_string}_BETA"
                                                                                                          )

    test_data.loc[valid_test_idx, f"ACMGLLR_{condition_string}"] = ACMG_scores
    test_data.loc[valid_test_idx, f"P-KNN_Pathogenic_{condition_string}"] = P_KNN_pathogenic

    combine_data = pd.concat([combine_data, evidence_strength_data], ignore_index=True)

    mean_evidence_strength = evidence_strength_data[evidence_strength_data['Label']=='Pathogenic variants']['Score'].mean() + evidence_strength_data[evidence_strength_data['Label']=='Benign variants']['Score'].mean()
    if mean_evidence_strength >  best_mean_evidence_strength:
        best_mean_evidence_strength = mean_evidence_strength
        best_tool = i
        
    print("pathogenic evidence mean", f"{evidence_strength_data[evidence_strength_data['Label']=='Pathogenic variants']['Score'].mean():.3f}")
    print("benign evidence mean", f"{evidence_strength_data[evidence_strength_data['Label']=='Benign variants']['Score'].mean():.3f}")

print(f"Best Tool {select_column[best_tool]}")

# Deal with miscalibration

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# tool="MutPred2.0_score"
tool="BayesDel_nsfp33a_noAF"
# tool="VEST4_score"
# tool="REVEL_score"

upper = 0.3
lower = 0.2

x = test_data[tool]
y = test_data[f'P-KNN_Pathogenic_{tool}']

plt.figure(figsize=(6, 6))
plt.scatter(x, y, alpha=0.6, c='purple', label='Variants')
plt.xlabel(tool)
plt.ylabel(f'Posterior Probability Pathogenic_{tool}')
plt.title(f'Scatter Plot of {tool} vs. P-KNN')
plt.grid(True)

idx_05 = (np.abs(y - upper)).idxmin()
idx_06 = (np.abs(y - lower)).idxmin()

x_05 = x.loc[idx_05]
x_06 = x.loc[idx_06]

plt.scatter([x_05], [y.loc[idx_05]], color='orange', label='Closest to 0.5', zorder=5)
plt.scatter([x_06], [y.loc[idx_06]], color='cyan', label='Closest to 0.6', zorder=5)

plt.text(x_05, y.loc[idx_05], f"{x_05:.3f}", fontsize=9, ha='right')
plt.text(x_06, y.loc[idx_06], f"{x_06:.3f}", fontsize=9, ha='right')

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import copy

for i in [0]:
    select_feature = i
    valid_calibration_idx = ~np.isnan(calibration_feature[:, select_feature])
    calibration_array = calibration_feature[valid_calibration_idx][:, select_feature].reshape(-1, 1)
    calibration_label = copy.deepcopy(calibration_label_bk[valid_calibration_idx])

    valid_test_idx = ~np.isnan(test_feature[:, select_feature])
    test_array = test_feature[valid_test_idx][:, select_feature].reshape(-1, 1)
    test_label = copy.deepcopy(test_label_bk[valid_test_idx])

    print(f"Tool {select_column[i]}, calibration size: {calibration_array.shape[0]}, test size: {test_array.shape[0]}")

    red_mask_calibration = calibration_label == 1
    blue_mask_calibration = calibration_label == 0
    red_mask_test = test_label == 1
    blue_mask_test = test_label == 0

    combined_values = np.concatenate([
        calibration_array[red_mask_calibration].ravel(),
        calibration_array[blue_mask_calibration].ravel(),
        test_array[red_mask_test].ravel(),
        test_array[blue_mask_test].ravel()
    ])
    bins = np.histogram_bin_edges(combined_values, bins=30)

    colors = {
        'Calibration-Pathogenic': 'red',
        'Calibration-Benign': 'blue',
        'Test-Pathogenic': '#cc6666',
        'Test-Benign': '#339999',
    }

    fig, ax = plt.subplots(figsize=(5, 6))
    ax.hist(calibration_array[red_mask_calibration], bins=bins, color=colors['Calibration-Pathogenic'],
            histtype='stepfilled', alpha=0.6, label='Calibration-pathogenic variants', density=True)
    ax.hist(calibration_array[blue_mask_calibration], bins=bins, color=colors['Calibration-Benign'],
            histtype='stepfilled', alpha=0.6, label='Calibration-benign variants', density=True)
    ax.hist(test_array[red_mask_test], bins=bins, color=colors['Test-Pathogenic'],
            histtype='stepfilled', alpha=0.3, label='Test-pathogenic variants', density=True)
    ax.hist(test_array[blue_mask_test], bins=bins, color=colors['Test-Benign'],
            histtype='stepfilled', alpha=0.3, label='Test-benign variants', density=True)

    ax.set_xlabel("Feature Value")
    ax.set_ylabel("Density")
    ax.set_xticks([-1, -0.5, 0, 0.5])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    if select_column[i] == 'BayesDel_nsfp33a_noAF':
        ax.legend(loc='upper left', fontsize=14, bbox_to_anchor=(0, 1.5))
    else:
        ax.legend(loc='upper center')

    if select_column[i] == 'BayesDel_nsfp33a_noAF':
        x_raw = test_data['BayesDel_nsfp33a_noAF'].values
        x_knn = test_data['P-KNN_Pathogenic_BayesDel_nsfp33a_noAF'].values
        valid_mask = ~np.isnan(x_raw) & ~np.isnan(x_knn)
        x_raw = x_raw[valid_mask]
        x_knn = x_knn[valid_mask]

        desired_knn_scores = np.array([0.0, 0.01, 0.1, 0.4, 0.8])
        matched_raw = []
        for s in desired_knn_scores:
            idx = np.abs(x_knn - s).argmin()
            matched_raw.append(x_raw[idx])

        ax_bottom = ax.twiny()
        ax_bottom.set_xlim(ax.get_xlim())
        ax_bottom.set_xticks(matched_raw)
        ax_bottom.set_xticklabels([f"{v:.2f}" if v < 0.1 else f"{v:.1f}" for v in desired_knn_scores])
        ax_bottom.set_xlabel("P-KNN Pathogenic Probability (BayesDel_nsfp33a_noAF)", fontsize=11)

        ax_bottom.xaxis.set_ticks_position('bottom')
        ax_bottom.xaxis.set_label_position('bottom')
        ax_bottom.spines['top'].set_visible(False)
        ax_bottom.spines['right'].set_visible(False)
        ax_bottom.spines['bottom'].set_position(('outward', 60))

    plt.tight_layout()
    plt.savefig(f"{select_column[i]}_distribution_with_calibrated_ticks.svg", format="svg", bbox_inches='tight')
    plt.show()

In [ ]:
import torch
import gc
from P_KNN_GPU import get_bootstrap_KNN_score_gpu, get_P_KNN_ACMG_score_1D, evaluate_result_1D
import copy
import os

#Parameter setting
Pprior = 0.0441; # Prior probability of pathogenicity (changes w/ c)
w_calibration=None
n_calibration_in_window=100;   # minimum number of clinvar variants in a local window
frac_regularization_in_window=0.03
batch_size = 512 # for 16GB VRAM, use 512
normalization= None
impute = False
mi_scaling = False
n_bootstrap = 100

p_value = 0.05
logbase = 1124

# combine_data = pd.DataFrame([])

best_mean_evidence_strength = 0

for i in range(4):   
    condition_string = select_column[i]
    select_feature = i

    valid_calibration_idx = ~np.isnan(calibration_feature[:,select_feature]) 
    calibration_array = calibration_feature[valid_calibration_idx][:,select_feature].reshape(-1, 1)
    calibration_label = copy.deepcopy(calibration_label_bk[valid_calibration_idx])

    test_array = calibration_array
    test_label = copy.deepcopy(calibration_label)

    valid_regularization_idx = ~np.isnan(regularization_feature[:,select_feature])
    regularization_array = regularization_feature[valid_regularization_idx][:,select_feature].reshape(-1, 1)

    print("")
    print(f"Tool {condition_string}")
    print(f"calibration size: {calibration_array.shape[0]}, test size: {test_array.shape[0]}")

    torch.cuda.empty_cache()
    gc.collect()

    # test_results_array = get_bootstrap_KNN_score_gpu(calibration_array, test_array, regularization_array, 
    #                                                  calibration_label, Pprior, w_calibration, 
    #                                                  n_calibration_in_window, frac_regularization_in_window, 
    #                                                  normalization, impute, mi_scaling, n_bootstrap, batch_size)

    # np.save(os.path.join(root_path, 'PKNNMatrix', f'PKNNMatrix_calibrateset_{condition_string}.npy'), test_results_array)

    test_results_array = np.load(os.path.join(root_path, 'PKNNMatrix', f'PKNNMatrix_calibrateset_{condition_string}.npy'))

    P_KNN_pathogenic, P_KNN_benign, ACMG_scores = get_P_KNN_ACMG_score_1D(test_results_array, test_array, p_value, Pprior, logbase)

    evidence_strength_data, pathogenic_calibration_dict, benign_calibration_dict = evaluate_result_1D(test_results_array,
                                                                                                      test_array,
                                                                                                      test_label, 
                                                                                                      p_value, 
                                                                                                      Pprior, 
                                                                                                      logbase, 
                                                                                                      category = condition_string, 
                                                                                                      show_plot=True, 
                                                                                                      save_name=None
                                                                                                          )

    calibration_data.loc[valid_calibration_idx, f"ACMGLLR_{condition_string}"] = ACMG_scores
    calibration_data.loc[valid_calibration_idx, f"P-KNN_Pathogenic_{condition_string}"] = P_KNN_pathogenic

    # combine_data = pd.concat([combine_data, evidence_strength_data], ignore_index=True)

    mean_evidence_strength = evidence_strength_data[evidence_strength_data['Label']=='Pathogenic variants']['Score'].mean() + evidence_strength_data[evidence_strength_data['Label']=='Benign variants']['Score'].mean()
    if mean_evidence_strength >  best_mean_evidence_strength:
        best_mean_evidence_strength = mean_evidence_strength
        best_tool = i
        
    print("pathogenic evidence mean", f"{evidence_strength_data[evidence_strength_data['Label']=='Pathogenic variants']['Score'].mean():.3f}")
    print("benign evidence mean", f"{evidence_strength_data[evidence_strength_data['Label']=='Benign variants']['Score'].mean():.3f}")

print(f"Best Tool {select_column[best_tool]}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import copy

Pprior = 0.0441

for i in [0]:
    if i=='P_KNN_pathogenic':
        tool = i
        score_tool = i
    else:    
        tool = select_column[i]
        score_tool = f"P-KNN_Pathogenic_{tool}"

    calibration_pathogenic = calibration_data[calibration_data['ClinVar_annotation'] == 1][score_tool].dropna()
    calibration_benign = calibration_data[calibration_data['ClinVar_annotation'] == 0][score_tool].dropna()
    test_pathogenic = test_data[test_data['ClinVar_annotation'] == 1][score_tool].dropna()
    test_benign = test_data[test_data['ClinVar_annotation'] == 0][score_tool].dropna()

    bins = np.linspace(0,1,11)
    bin_centers = 0.5 * (bins[1:] + bins[:-1])

    cal_path_dens, _ = np.histogram(calibration_pathogenic, bins=bins, density=True)
    test_path_dens, _ = np.histogram(test_pathogenic, bins=bins, density=True)
    cal_ben_dens, _ = np.histogram(calibration_benign, bins=bins, density=True)
    test_ben_dens, _ = np.histogram(test_benign, bins=bins, density=True)

    calibration_prob = cal_path_dens*Pprior/(cal_path_dens*Pprior+cal_ben_dens*(1-Pprior))
    test_prob = test_path_dens*Pprior/(cal_path_dens*Pprior+test_ben_dens*(1-Pprior))

    fig, ax = plt.subplots(figsize=(6, 3))
    
    ax.plot(bin_centers, calibration_prob, color='purple', label='Calibration dataset \nnormalized % of pathogenic variants', linewidth=2, alpha=0.7)
    ax.plot(bin_centers, test_prob, color='orchid', label='Test dataset \nnormalized % of pathogenic variants', linewidth=2, alpha=0.7)
    ax.set_ylabel("Normalized % of \npathogenic variants", fontsize=12)
    ax.set_xlim([0, 1.05])
    ax.set_xlabel("Posterior probability (pathogenic)")
    
    ax.plot(bin_centers, calibration_prob-test_prob, color='orange', label='Difference (calibration - test) \nnormalized % of pathogenic variants', linewidth=4, alpha=0.7)
    
    ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.5), fontsize=14)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f"{select_column[i]}_densityratio.svg", format="svg", bbox_inches='tight')

    plt.show()

In [ ]:
def weighted_score_with_binom_ci(p_array, n_array, w, p_value=0.05):
    from scipy.stats import binom
    p_array = np.asarray(p_array)
    n_array = np.asarray(n_array)
    shape = p_array.shape

    scores = np.full(shape, np.nan)
    ci_lower = np.full(shape, np.nan)
    ci_upper = np.full(shape, np.nan)

    t_array = p_array + n_array

    it = np.nditer(p_array, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        p = p_array[idx]
        t = t_array[idx]

        if t > 0:
            pi_hat = p / t
            ci_low, ci_up = binom.interval(1-p_value, t, pi_hat)
            
            def weighted(pi):
                return pi / (pi + w * (1 - pi))

            scores[idx] = weighted(pi_hat)
            ci_lower[idx] = weighted(ci_low / t)
            ci_upper[idx] = weighted(ci_up / t)
        
        it.iternext()
    
    return scores, ci_lower, ci_upper

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="ticks")
import numpy as np
import copy
from scipy.interpolate import interp1d

Pprior = 0.0441
logbase = 1124

Post_p = np.zeros(4) 
Post_b = np.zeros(4)

for j in range(4):
    Post_p[j] = logbase ** (1 / 2 ** j) * Pprior / ((logbase ** (1 / 2 ** j) - 1) * Pprior + 1)
    Post_b[j] = (logbase ** (1 / 2 ** j)) * (1 - Pprior) / (((logbase ** (1 / 2 ** j)) - 1) * (1 - Pprior) + 1)

for i in range(4):#, 'P_KNN_pathogenic']:
    if i=='P_KNN_pathogenic':
        tool = i
        score_tool = i
    else:    
        tool = select_column[i]
        score_tool = f"P-KNN_Pathogenic_{tool}"

    calibration_pathogenic = calibration_data[calibration_data['ClinVar_annotation'] == 1][score_tool].dropna()
    calibration_benign = calibration_data[calibration_data['ClinVar_annotation'] == 0][score_tool].dropna()
    test_pathogenic = test_data[test_data['ClinVar_annotation'] == 1][score_tool].dropna()
    test_benign = test_data[test_data['ClinVar_annotation'] == 0][score_tool].dropna()

    w_test = (1 - Pprior) * len(test_pathogenic) / (len(test_benign) * Pprior) 

    bins = np.linspace(0,1,11)
    bin_centers = 0.5 * (bins[1:] + bins[:-1])

    with np.errstate(divide='ignore', invalid='ignore'):
        ratio_path_dens = np.where(cal_path_dens != 0, test_path_dens / cal_path_dens, np.nan)
        ratio_ben_dens = np.where(cal_ben_dens != 0, test_ben_dens / cal_ben_dens , np.nan)

    pathogenic_counts, _ = np.histogram(test_pathogenic, bins=bins, density=False)
    benign_counts, _ = np.histogram(test_benign, bins=bins, density=False)

    pathogenic_ratios, ci_lower, ci_upper = weighted_score_with_binom_ci(pathogenic_counts, benign_counts, w_test, p_value=0.05)
    valid_mask = ~np.isnan(pathogenic_ratios) & ~np.isnan(ci_lower) & ~np.isnan(ci_upper)

    cal_path_dens, _ = np.histogram(calibration_pathogenic, bins=bins, density=True)
    test_path_dens, _ = np.histogram(test_pathogenic, bins=bins, density=True)
    cal_ben_dens, _ = np.histogram(calibration_benign, bins=bins, density=True)
    test_ben_dens, _ = np.histogram(test_benign, bins=bins, density=True)

    calibration_prob = cal_path_dens*Pprior/(cal_path_dens*Pprior+cal_ben_dens*(1-Pprior))
    test_prob = test_path_dens*Pprior/(cal_path_dens*Pprior+test_ben_dens*(1-Pprior))

    valid_mask = ~np.isnan(pathogenic_ratios) & ~np.isnan(ci_lower) & ~np.isnan(ci_upper)

    fig, ax1 = plt.subplots(figsize=(6, 5.45))

    ax1.plot(bin_centers[valid_mask], pathogenic_ratios[valid_mask], 
             'o-', color='red', label='Frequency of pathogenic variants')
    ax1.fill_between(bin_centers[valid_mask], 
                     ci_lower[valid_mask], ci_upper[valid_mask], 
                     color='red', alpha=0.2, label='95% Confidence Interval')
    ax1.set_ylabel("Pathogenic Probability", fontsize=12)
    ax1.tick_params(axis='y')
    
    for threshold in Post_p:
        ax1.axvline(x=threshold, color='orange', linestyle='--', alpha=0.2)
        ax1.axhline(y=threshold, color='orange', linestyle='--', alpha=0.2)
    
    x_vals = np.linspace(0, 1, 100)
    ax1.plot(x_vals, x_vals, color='red', linestyle='--', alpha=0.2)
    
    ax2 = ax1.twinx()
    ax2.plot(bin_centers, calibration_prob - test_prob, color='orange', linestyle='-', label='Difference (calibration - test) \nnormalized % of pathogenic variants', linewidth=3, alpha = 0.7)
    ax2.set_ylabel("Normalized % of pathogenic variants", fontsize=12)
    y_max = np.nanmax(calibration_prob - test_prob) 
    ax2.set_yticks(np.arange(0, y_max + 0.1, 0.1))
    ax2.tick_params(axis='y')
    
    ax1.set_xlabel("P-KNN Pathogenic Score", fontsize=12)
    ax1.set_xlim([0, 1.0])

    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
        
    all_handles = handles1 + handles2
    all_labels = labels1 + labels2
    
    fig.legend(all_handles, all_labels, loc='lower center', bbox_to_anchor=(0.4, 1.0), frameon=True)

    plt.tight_layout()
    plt.savefig(f"{tool}_combined_dualyaxis.svg", 
            format="svg", bbox_inches='tight')
    plt.show()

# Integrating All tools

In [ ]:
import copy

calibration_feature = calibration_data[select_column[:-1]].to_numpy()
test_feature = test_data[select_column[:-1]].to_numpy()
regularization_feature = gnomAD_set[select_column[:-1]].to_numpy()

calibration_label = calibration_data[select_column[-1]].to_numpy()
calibration_label_bk = copy.deepcopy(calibration_label)
test_label = test_data[select_column[-1]].to_numpy()
test_label_bk = copy.deepcopy(test_label)

print(calibration_feature.shape, test_feature.shape, regularization_feature.shape)

In [ ]:
import torch
import gc
from P_KNN_GPU import get_bootstrap_KNN_score_gpu, get_P_KNN_ACMG_score, evaluate_result
import copy

#Parameter setting
Pprior = 0.0441; # Prior probability of pathogenicity (changes w/ c)
w_calibration = None
n_calibration_in_window = 100   # minimum number of clinvar variants in a local window
frac_regularization_in_window = 0.03
batch_size = 512 # This is for A100, if V100, use 512
normalization= 'rank'
impute = True
mi_scaling = True
n_bootstrap = 100
version=53

p_value = 0.05
logbase = 1124

# combine_data = pd.DataFrame([])

condition_string = f'P-KNN'
print(condition_string)
valid_calibration_idx = ~np.isnan(calibration_feature).all(axis=1) 
calibration_array = calibration_feature[valid_calibration_idx]
calibration_label = copy.deepcopy(calibration_label_bk[valid_calibration_idx])

valid_test_idx = ~np.isnan(test_feature).all(axis=1) 
test_array = test_feature[valid_test_idx]
test_label = copy.deepcopy(test_label_bk[valid_test_idx])

valid_regularization_idx = ~np.isnan(regularization_feature).all(axis=1) 
regularization_array = regularization_feature[valid_regularization_idx]

print(f"calibration size: {calibration_array.shape[0]}, test size: {test_array.shape[0]}")

torch.cuda.empty_cache()
gc.collect()

# test_results_array = get_bootstrap_KNN_score_gpu(calibration_array, test_array, regularization_array, 
#                                                  calibration_label, Pprior, w_calibration, 
#                                                  n_calibration_in_window, frac_regularization_in_window, 
#                                                  normalization, impute, mi_scaling, n_bootstrap, batch_size)

# np.save(os.path.join(root_path, 'PKNNMatrix', f'PKNNMatrix_{condition_string}.npy'), test_results_array)

test_results_array = np.load(os.path.join(root_path, 'PKNNMatrix', f'PKNNMatrix_{condition_string}.npy'))


P_KNN_pathogenic, P_KNN_benign, ACMG_scores = get_P_KNN_ACMG_score(test_results_array, p_value, Pprior, logbase)

test_data.loc[valid_test_idx, f"ACMGLLR"] = ACMG_scores 

evidence_strength_data, pathogenic_calibration_dict, benign_calibration_dict = evaluate_result(test_results_array,
                                                                                                  test_label, 
                                                                                                  p_value, 
                                                                                                  Pprior, 
                                                                                                  logbase, 
                                                                                                  category = condition_string, 
                                                                                                  show_plot=True, 
                                                                                                  save_name='P_KNN')

combine_data = pd.concat([combine_data, evidence_strength_data], ignore_index=True)
    
print("pathogenic evidence mean", f"{evidence_strength_data[evidence_strength_data['Label']=='Pathogenic variants']['Score'].mean():.3f}")
print("benign evidence mean", f"{evidence_strength_data[evidence_strength_data['Label']=='Benign variants']['Score'].mean():.3f}")

In [ ]:
combine_data['Category'].unique()

In [ ]:
# 1. 定義你想要的順序 (對應 combine_data['Category'] 裡原始的值)
my_order = ['P-KNN', 
            'VARITY_R_score', 
            'AlphaMissense_score', 
            'BayesDel_nsfp33a_noAF',
            'REVEL_score',
            'VEST4_score',
            'MutPred2.0_score',
            'ESM1b_score'
           ]

# 2. 定義原始名稱與新名稱的對應
name_map = {
    'BayesDel_nsfp33a_noAF': 'BayesDel',
    'MutPred2.0_score': 'MutPred2',
    'REVEL_score': 'REVEL',
    'VEST4_score': 'VEST4', 
    'AlphaMissense_score': 'AlphaMissense',
    'ESM1b_score': 'ESM1b',
    'VARITY_R_score': 'VARITY_R',
    'P-KNN': 'P-KNN'
}

# 3. 產生新的 Tick Labels (照著 my_order 的順序換名)
new_labels = [name_map.get(x, x) for x in my_order]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

## violin plot
sns.set(style="ticks")
plt.figure(figsize=(14, 4))
ax = sns.violinplot(
    x="Category",
    y="Score", 
    hue="Label",   # Pathogenic and Benign 
    data=combine_data, 
    order=my_order,
    split=True,   # on each side of violin
    inner="box", 
    palette={"Pathogenic variants": "red", "Benign variants": "blue"},
    alpha=0.6, 
    density_norm='area',
    hue_order=['Pathogenic variants', 'Benign variants'],
    width=1.2
)

dark_gray = '0.3'

sns.boxplot(
    x="Category", y="Score", hue="Label", data=combine_data,
    order=my_order, 
    showcaps=True,  
    showfliers=False,
    palette={"Pathogenic variants": "pink", "Benign variants": "#bae0f5"},
    width=0.1, dodge=True, ax=ax,
    whiskerprops={'color': dark_gray, 'linewidth': 1.5, 'zorder': 2},
    capprops={'color': dark_gray, 'linewidth': 1.5},
    medianprops={'color': dark_gray, 'linewidth': 1.5},
    boxprops={'zorder': 2, 'edgecolor': dark_gray, 'linewidth': 1.5},
    hue_order=['Pathogenic variants', 'Benign variants']

)

handles, labels = ax.get_legend_handles_labels()
n_hue = combine_data["Label"].nunique() 
ax.legend(handles[:n_hue], labels[:n_hue], loc="upper right", fontsize=12)
ax.set_xticklabels(new_labels, fontsize=14)
plt.xlabel("")
plt.xticks(fontsize=14)
plt.ylabel("Evidence strength (LLR)", fontsize=14)
plt.yticks([-6, -4, -2, 0, 2, 4, 6, 8])
plt.ylim([-7, 10])
sns.despine(top=True, right=True)

plt.savefig("Fig3A.svg", format="svg")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns

# ── 0. 設定 ───────────────────────────────────────────────────────────
name_map = {
    'BayesDel_nsfp33a_noAF': 'BayesDel',
    'MutPred2.0_score':       'MutPred2',
    'REVEL_score':            'REVEL',
    'VEST4_score':            'VEST4',
    'AlphaMissense_score':    'AlphaMissense',
    'ESM1b_score':            'ESM1b',
    'VARITY_R_score':         'VARITY_R',
    'P-KNN':                  'P-KNN'
}
new_labels = [name_map.get(x, x) for x in my_order]

benign_bins     = [(-np.inf,-8), (-8,-4), (-4,-3), (-3,-2), (-2,-1)]
pathogenic_bins = [(1,2), (2,3), (3,4), (4,8), (8,np.inf)]
bin_labels = [
    "BP4_VeryStrong(-8)", "BP4_Strong(-4)", "BP4_3(-3)",
    "BP4_Moderate(-2)", "BP4_Supporting(-1)",
    "",
    "PP3_Supporting(+1)", "PP3_Moderate(+2)", "PP3_3(+3)",
    "PP3_Strong(+4)", "PP3_VeryStrong(+8)"
]
all_bins = benign_bins + [None] + pathogenic_bins

# ── 字體大小（統一在這裡調）─────────────────────────────────────────
annot_fs = 18
xtick_fs = 18
ytick_fs = 18
cbar_fs  = 12
title_fs = 14

# ── 1. 還原 benign score ─────────────────────────────────────────────
df = combine_data.copy()
df.loc[df["Label"] == "Benign variants", "Score"] *= -1

# ── 2. 計算 LR ───────────────────────────────────────────────────────
records = []
for tool in my_order:
    sub = df[df["Category"] == tool].dropna(subset=["Score"])
    n_path   = (sub["Label"] == "Pathogenic variants").sum()
    n_benign = (sub["Label"] == "Benign variants").sum()
    for bin_range, col_label in zip(all_bins, bin_labels):
        if bin_range is None:
            records.append({"Tool": tool, "Bin": col_label, "LR": np.nan,
                            "is_gap": True, "n_p": 0, "n_b": 0})
            continue
        lo, hi = bin_range
        mask = (sub["Score"] > lo) & (sub["Score"] <= hi)
        n_p = ((sub["Label"] == "Pathogenic variants") & mask).sum()
        n_b = ((sub["Label"] == "Benign variants")     & mask).sum()
        p_p = n_p / n_path   if n_path   > 0 else np.nan
        p_b = n_b / n_benign if n_benign > 0 else np.nan
        lr  = p_p / p_b if (p_b and p_b > 0) else np.nan
        records.append({"Tool": tool, "Bin": col_label, "LR": lr,
                        "is_gap": False, "n_p": n_p, "n_b": n_b})

lr_df     = pd.DataFrame(records)
lr_pivot  = lr_df.pivot(index="Tool", columns="Bin", values="LR")[bin_labels].reindex(my_order)
gap_pivot = lr_df.pivot(index="Tool", columns="Bin", values="is_gap")[bin_labels].reindex(my_order)

# ── 3. Annotation 文字 ───────────────────────────────────────────────
annot = lr_pivot.copy().astype(object)
for r in lr_pivot.index:
    for c in lr_pivot.columns:
        val = lr_pivot.loc[r, c]
        if gap_pivot.loc[r, c] or pd.isna(val):
            annot.loc[r, c] = ""
        else:
            annot.loc[r, c] = f"{val:.2f}"

# ── 4. Colorscale ────────────────────────────────────────────────────
log_lr  = np.log10(lr_pivot.astype(float))
valid   = log_lr.values[~np.isnan(log_lr.values) & ~gap_pivot.values.astype(bool)]
vmax    = max(np.nanpercentile(np.abs(valid), 95), 0.01)
norm    = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

# ── 5. 圖形尺寸 ──────────────────────────────────────────────────────
n_cols, n_rows = len(bin_labels), len(my_order)
cell  = 1.1
fig_w = n_cols * cell + 4
fig_h = n_rows * cell + 2.5

fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.set_facecolor("white")

# ── 6. 主 heatmap ────────────────────────────────────────────────────
sns.heatmap(
    log_lr,
    ax=ax,
    cmap="RdBu_r",
    norm=norm,
    annot=annot,
    fmt="",
    linewidths=0.5,
    linecolor="white",
    annot_kws={"size": annot_fs},
    cbar_kws={"label": "log10(LR)", "shrink": 0.6},
    mask=log_lr.isna(),
)

# ── 7. 空白格子加框線 ─────────────────────────────────────────────────
gap_col_idx = bin_labels.index("")
for row_i in range(n_rows):
    ax.add_patch(mpatches.Rectangle(
        (gap_col_idx, row_i), 1, 1,
        facecolor="#d3d3d3", edgecolor="white",
        linewidth=0.5, zorder=2
    ))
    for col_i, col_label in enumerate(bin_labels):
        if col_label == "":
            continue
        if pd.isna(lr_pivot.iloc[row_i, col_i]):
            ax.add_patch(mpatches.Rectangle(
                (col_i, row_i), 1, 1,
                facecolor="white", edgecolor="black",
                linewidth=0.8, zorder=2
            ))

# ── 8. 最外框 ────────────────────────────────────────────────────────
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor("black")
    spine.set_linewidth(1.5)

# ── 9. Labels & 字體 ─────────────────────────────────────────────────
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_xticklabels(bin_labels, fontsize=xtick_fs, rotation=45, ha="right")
ax.set_yticklabels(new_labels,  fontsize=ytick_fs, rotation=0)
ax.tick_params(axis='both', labelsize=ytick_fs)

for tk in ax.get_xticklabels():
    if tk.get_text() == "":
        tk.set_visible(False)

# colorbar 字體
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=cbar_fs)
cbar.set_label("log10(LR)", fontsize=cbar_fs)

ax.set_title("ClinGen-style LR heatmap (PP3/BP4)", fontsize=title_fs, pad=12)
plt.tight_layout()
plt.savefig("Fig3B_heatmap.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="ticks")
fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(6, 4))

dark_gray = '0.3'
palette_violin = {"Pathogenic variants": "red",  "Benign variants": "blue"}
palette_box    = {"Pathogenic variants": "pink", "Benign variants": "#bae0f5"}

demo_cat = my_order[0]
demo_data = combine_data[combine_data["Category"] == demo_cat].copy()

left_data = demo_data.copy()
left_data.loc[left_data["Label"] == "Benign variants", "Score"] *= -1
right_data = demo_data.copy()

for ax, data, hue_ord in [
    (ax_left,  left_data,  ['Pathogenic variants', 'Benign variants']),
    (ax_right, right_data, ['Benign variants', 'Pathogenic variants']),
]:
    sns.violinplot(
        x="Category", y="Score", hue="Label",
        data=data,
        split=True, inner="box",
        palette=palette_violin,
        alpha=0.6, density_norm='area',
        hue_order=hue_ord,
        ax=ax
    )
    sns.boxplot(
        x="Category", y="Score", hue="Label",
        data=data,
        showcaps=True, showfliers=False,
        palette=palette_box,
        width=0.1, dodge=True, ax=ax,
        whiskerprops={'color': dark_gray, 'linewidth': 1.5, 'zorder': 2},
        capprops={'color': dark_gray, 'linewidth': 1.5},
        medianprops={'color': dark_gray, 'linewidth': 1.5},
        boxprops={'zorder': 2, 'edgecolor': dark_gray, 'linewidth': 1.5},
        hue_order=hue_ord
    )
    ax.set_xlabel("")
    ax.set_xticklabels(["Tool"], fontsize=12)
    ax.legend_.remove()

# 左圖：正常左側 y 軸
ax_left.set_ylabel("ACMG score", fontsize=13)
sns.despine(ax=ax_left, top=True, right=True)

# 右圖：y 軸移到右側
ax_right.set_ylabel("Evidence strength", fontsize=13)
ax_right.yaxis.set_label_position("right")
ax_right.yaxis.tick_right()
sns.despine(ax=ax_right, top=True, left=True, right=False)

# legend 放整張圖的中下
handles, labels = ax_right.get_legend_handles_labels()
n_hue = demo_data["Label"].nunique()
fig.legend(
    handles[:n_hue], labels[:n_hue],
    loc="lower center",
    ncol=2,
    fontsize=12,
    bbox_to_anchor=(0.5, 0.13)
)

# y 軸範圍與刻度統一
y_min = min(ax_left.get_ylim()[0], ax_right.get_ylim()[0])
y_max = max(ax_left.get_ylim()[1], ax_right.get_ylim()[1])
for ax in (ax_left, ax_right):
    ax.set_ylim(y_min, y_max)
    ax.set_yticks([-8, -4, 0, 4, 8])

plt.tight_layout()
plt.savefig("Fig2A_schematic.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, auc
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

roc_curves = []

for col in select_column[:-1]+['ACMGLLR']:  # Exclude 'ClinVar_annotation'
    roc_curves.append(calculate_roc_and_auc(
        test_data['ClinVar_annotation'], test_data[col], col))

# plt.figure(figsize=(8, 6))
# for fpr, tpr, roc_auc, label in roc_curves:
#     plt.plot(fpr, tpr, label=f'{label} (AUC = {roc_auc:.2f})')

# plt.plot([0, 1], [0, 1], color='gray', linestyle='--')  # Reference diagonal line
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('ROC Curves')
# plt.legend(loc='lower right')
# # plt.grid()
# plt.show()
plt.figure(figsize=(6, 6))

for fpr, tpr, roc_auc, label in roc_curves:
    # 檢查是否為目標欄位
    if label == 'ACMGLLR':
        display_label = 'P_KNN'  # 修改圖例名稱
        line_width = 3           # 加粗
        line_color = 'red'       # 設定鮮明顏色（例如紅色）
        zorder = 5               # 確保這條線在最上層
    else:
        display_label = label
        line_width = 1.5         # 一般粗細
        line_color = None        # 使用預設配色循環
        zorder = 2

    plt.plot(fpr, tpr, label=f'{display_label} (AUC = {roc_auc:.2f})', 
             linewidth=line_width, color=line_color, zorder=zorder)

plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig("Fig2S_AUROC.svg", format="svg", bbox_inches="tight")
plt.show()


In [ ]:
# 1. 定義你想要的順序 (對應 combine_data['Category'] 裡原始的值)
my_order = ['P-KNN6', 
            # 'VARITY_R_score', 
            'AlphaMissense_score', 
            'BayesDel_nsfp33a_noAF',
            'REVEL_score',
            'VEST4_score',
            'MutPred2.0_score',
            'ESM1b_score'
           ]

# 2. 定義原始名稱與新名稱的對應
name_map = {
    'BayesDel_nsfp33a_noAF': 'BayesDel',
    'MutPred2.0_score': 'MutPred2',
    'REVEL_score': 'REVEL',
    'VEST4_score': 'VEST4', 
    'AlphaMissense_score': 'AlphaMissense',
    'ESM1b_score': 'ESM1b',
    # 'VARITY_R_score': 'VARITY_R',
    'P-KNN6': 'P-KNN'
}

# 3. 產生新的 Tick Labels (照著 my_order 的順序換名)
new_labels = [name_map.get(x, x) for x in my_order]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

## violin plot
sns.set(style="ticks")
plt.figure(figsize=(14, 3.5))
ax = sns.violinplot(
    x="Category",
    y="Score", 
    hue="Label",   # Pathogenic and Benign 
    data=combine_data, 
    order=my_order,
    split=True,   # on each side of violin
    inner="box", 
    palette={"Pathogenic variants": "red", "Benign variants": "blue"},
    alpha=0.6, 
    density_norm='area',
    hue_order=['Pathogenic variants', 'Benign variants'] 
)

dark_gray = '0.3'

sns.boxplot(
    x="Category", y="Score", hue="Label", data=combine_data,
    order=my_order, 
    showcaps=True,  
    showfliers=False,
    palette={"Pathogenic variants": "pink", "Benign variants": "#bae0f5"},
    width=0.1, dodge=True, ax=ax,
    whiskerprops={'color': dark_gray, 'linewidth': 1.5, 'zorder': 2},
    capprops={'color': dark_gray, 'linewidth': 1.5},
    medianprops={'color': dark_gray, 'linewidth': 1.5},
    boxprops={'zorder': 2, 'edgecolor': dark_gray, 'linewidth': 1.5},
    hue_order=['Pathogenic variants', 'Benign variants']

)

handles, labels = ax.get_legend_handles_labels()
n_hue = combine_data["Label"].nunique() 
ax.legend(handles[:n_hue], labels[:n_hue], loc="upper right", fontsize=12)
ax.set_xticklabels(new_labels, fontsize=14)
plt.xlabel("")
plt.xticks(fontsize=14)
plt.ylabel("Evidence strength (LLR)", fontsize=14)
plt.yticks([-6, -4, -2, 0, 2, 4, 6, 8])
plt.ylim([-7, 10])
sns.despine(top=True, right=True)

# plt.savefig("Fig5A.svg", format="svg")
plt.show()

# Comparison

In [ ]:
test_data.isna().sum()

In [ ]:
import pandas as pd
from scipy import stats

comparison_tool = 'VARITY_R_score'

for i in [0, 1]:
    print(i)
    df = test_data[test_data['ClinVar_annotation']==i][['ACMGLLR', comparison_tool]]
    df = df.dropna(subset=['ACMGLLR', comparison_tool])
    display(df)
    
    p_value_cutoff = 0.05
    
    side = 'greater' if i == 1 else 'less'

    t_stat, p_value = stats.ttest_rel(df['ACMGLLR'], df[comparison_tool], alternative=side)
    
    if p_value < p_value_cutoff and t_stat > 0:
        result = f"ACMGLLR is significantly higher to {comparison_tool}"
    elif p_value < p_value_cutoff and t_stat < 0:
        result = f"ACMGLLR is significantly lower to {comparison_tool}"
    else:
        result = "No significant difference between the two groups"
    
    print(f"T-statistic: {t_stat}")
    print(f"P-value: {np.format_float_scientific(p_value, precision=5)}")
    print(result)

In [ ]:
import pandas as pd
from scipy import stats

comparison_tool = 'AlphaMissense_score'

for i in [0, 1]:
    print(i)
    df = test_data[test_data['ClinVar_annotation']==i][['ACMGLLR', comparison_tool]]
    df = df.dropna(subset=['ACMGLLR', comparison_tool])
    display(df)
    
    p_value_cutoff = 0.05
    
    side = 'greater' if i == 1 else 'less'

    t_stat, p_value = stats.ttest_rel(df['ACMGLLR'], df[comparison_tool], alternative=side)
    
    if p_value < p_value_cutoff and t_stat > 0:
        result = f"ACMGLLR is significantly higher to {comparison_tool}"
    elif p_value < p_value_cutoff and t_stat < 0:
        result = f"ACMGLLR is significantly lower to {comparison_tool}"
    else:
        result = "No significant difference between the two groups"
    
    print(f"T-statistic: {t_stat}")
    print(f"P-value: {np.format_float_scientific(p_value, precision=5)}")
    print(result)